# 第八章 大型语言模型：进阶应用与挑战
----
书籍名称：《大数据分析：Python 实践与大型语言模型应用》

谢佳松、刘娟



----

## 8.1 大型语言模型简介与调用实践
### 8.1.3 大模型API调用实践：以DeepSeek为例

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
# 加载环境变量
env_file_path ="env/DeepSeek_KEY.env"
# 加载环境变量文件
load_dotenv(env_file_path)
# 从环境变量中读取 API Key
api_key = os.getenv("DeepSeek_KEY")

In [ ]:
#初始化 DeepSeek 客户端
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
# 构造请求
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "请用一句话总结AI的未来趋势。"}
    ],
    stream=False
)
print("模型回答：", response.choices[0].message.content)

In [ ]:
#  构造请求
response = client.chat.completions.create(
    model="deepseek-chat",  # 指定模型
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "请用一句话解释人工智能的核心概念。"}
    ],
    temperature=0.7,       # 温度参数，控制输出的随机性（0 = 确定性最大）
    max_tokens=200,        # 最大输出 token 数
    stream=False
)
#输出结果
print("模型回答：", response.choices[0].message.content)


In [ ]:
import asyncio
import os
from openai import AsyncOpenAI
from dotenv import load_dotenv
# 加载环境变量文件
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
# 从环境变量中读取 API Key
api_key = os.getenv("DeepSeek_KEY")
# 初始化 DeepSeek 异步客户端。注意：DeepSeek 需要指定 base_url
aclient = AsyncOpenAI(api_key=api_key, base_url="https://api.deepseek.com")
async def get_economic_explanation_async(concept: str) -> str:
    """
    异步获取经济学概念解释的函数。
    它是一个协程（coroutine），可以被 await。
    """
    try:
        # 使用 await 等待异步 API 调用完成
        response = await aclient.chat.completions.create(
            model="deepseek-chat", # 同样使用 DeepSeek 的聊天模型
            messages=[
                {"role": "system", "content": "你是一名专业的经济学教授，善于用清晰易懂的语言解释复杂的经济学概念。"},
                {"role": "user", "content": f"请用通俗的语言解释一下“{concept}”的概念。"}
            ],
            temperature=0.7,
            max_tokens=150,
            stream=False
        )
        if response.choices:
            return f"助手（{concept}）：{response.choices[0].message.content}"
        else:
            return f"未收到关于{concept}的回复。"
    except Exception as e:
        return f"获取{concept}时发生错误: {e}"
async def main():
    """
    主异步函数，用于协调多个异步任务。
    """
    concepts = ["机会成本", "边际效用", "供需曲线", "比较优势", "国民生产总值"]
    
    # 创建一系列异步任务 (coroutines)，但它们此时还未开始执行
    tasks = [get_economic_explanation_async(concept) for concept in concepts]
    
    print("开始并发获取解释...")
    # asyncio.gather 并发地运行 tasks 列表中的所有协程。
    # 它会等待所有协程都完成后，将它们的结果以列表的形式返回。
    results = await asyncio.gather(*tasks) # *tasks 解包列表，将每个协程作为单独的参数传入

    print("\n--- 所有解释已获取 ---")
    for result in results:
        print(result)
if __name__ == "__main__":
    # 运行主异步函数。这是启动 asyncio 事件循环的入口点。
    await main()

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# 加载环境变量文件
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
api_key = os.getenv("DeepSeek_KEY")

# 初始化 DeepSeek 客户端
# 注意：DeepSeek 需要指定 base_url
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

def get_llm_response(task_type: str, user_query: str) -> str:
    """
    根据任务类型选择合适的LLM模型进行调用。
    
    参数:
    task_type (str): 任务类型，例如 "creative_writing", "summary", "code_generation"。
    user_query (str): 用户输入的查询或内容。

    返回:
    str: 模型生成的响应文本，或错误信息。
    """
    model_configs = {
        # 文本生成和创意写作: 倾向于选择更强大的、具有高创造性的模型
        "creative_writing": "deepseek-reasoner", 
        # 总结和信息提取: 可以选择成本较低、响应速度更快的模型
        "summary": "deepseek-chat", 
        # 代码生成: DeepSeek 的代码模型
        "code_generation": "deepseek-coder", 
        # 默认通用模型
        "general": "deepseek-chat"
    }

    # 根据任务类型选择模型，如果未找到则默认为 "general"
    selected_model = model_configs.get(task_type, "general") 

    try:
        # 调用 DeepSeek API
        response = client.chat.completions.create(
            model=selected_model,
            messages=[
                {"role": "system", "content": "你是一名有用的助手。"}, # 系统角色设定
                {"role": "user", "content": user_query} # 用户查询
            ],
            temperature=0.7, # 控制生成文本的随机性
            max_tokens=200 # 限制生成文本的最大token数量
        )
        
        # 检查并返回模型响应
        if response.choices:
            return response.choices[0].message.content
        else:
            return "未收到模型回复。"
    except Exception as e:
        # 捕获并返回API调用中的错误
        return f"调用API时发生错误: {e}"

# 示例调用
if __name__ == "__main__":
    print("\n--- 任务类型选择示例 ---")
    # 创意写作示例，使用 deepseek-reasoner
    print("创意写作：", get_llm_response("creative_writing", "写一首关于经济学之美的诗歌。"))
    # 文本总结示例，使用 deepseek-chat
    print("文本总结：", get_llm_response("summary", "请总结一下二战后全球经济格局的主要变化。"))
    # 代码生成示例，使用 deepseek-coder
    print("代码生成：", get_llm_response("code_generation", "写一个Python函数，用于计算斐波那契数列。"))

In [ ]:
import time
import random
import os
from openai import OpenAI, APIError, RateLimitError, AuthenticationError, BadRequestError
from dotenv import load_dotenv

# 加载环境变量文件
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
api_key = os.getenv("DeepSeek_KEY")

# 初始化 DeepSeek 客户端
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

def call_llm_with_retry(
    messages: list, 
    model: str = "deepseek-chat", 
    max_retries: int = 5, 
    initial_delay: int = 1,
    temperature: float = 0.7,
    max_tokens: int = 200
) -> str:
    """
    调用LLM并包含健壮的错误处理和指数退避重试机制。

    参数:
    messages (list): 传递给模型的对话消息列表。
    model (str): 要使用的LLM模型名称。
    max_retries (int): 最大重试次数。
    initial_delay (int): 第一次重试前的初始等待秒数。
    temperature (float): 控制生成文本的随机性。
    max_tokens (int): 限制生成文本的最大token数量。

    返回:
    str: 模型生成的响应文本，或错误信息。
    """
    for i in range(max_retries):
        try:
            print(f"尝试调用模型 ({model})，第 {i+1}/{max_retries} 次...")
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            if response.choices:
                print("模型调用成功。")
                return response.choices[0].message.content
            else:
                print(f"第{i+1}次尝试: 未收到模型有效回复。")
                # 如果模型返回空 choices，可能是暂时性问题，可以重试
                if i < max_retries - 1:
                    delay = initial_delay * (2 ** i) + random.uniform(0, 1)
                    print(f"等待 {delay:.2f} 秒后重试...")
                    time.sleep(delay)
                else:
                    return "经过多次重试仍未收到模型有效回复。"

        except RateLimitError as e:
            # 捕获速率限制错误
            delay = initial_delay * (2 ** i) + random.uniform(0, 1)
            print(f"第{i+1}次尝试: 达到速率限制 ({e})，等待 {delay:.2f} 秒后重试...")
            time.sleep(delay)
        except AuthenticationError as e:
            # 捕获认证错误，这类错误通常不可重试，应立即报告
            print(f"认证错误: API 密钥无效或权限不足。请检查您的 DeepSeek_KEY。错误详情: {e}")
            return f"认证失败，请检查API密钥: {e}"
        except BadRequestError as e:
            # 捕获请求参数错误，这类错误通常是代码或输入问题，不可重试
            print(f"请求参数错误: 您的请求参数不合法。请检查 messages 内容或 max_tokens 等参数。错误详情: {e}")
            return f"请求参数错误: {e}"
        except APIError as e:
            # 捕获其他通用API错误，可能是服务器内部错误，可以重试
            delay = initial_delay * (2 ** i) + random.uniform(0, 1)
            print(f"第{i+1}次尝试: API 错误 ({e})，等待 {delay:.2f} 秒后重试...")
            time.sleep(delay)
        except Exception as e:
            # 捕获所有其他未预期的错误
            delay = initial_delay * (2 ** i) + random.uniform(0, 1)
            print(f"第{i+1}次尝试: 发生未知错误 ({e})，等待 {delay:.2f} 秒后重试...")
            time.sleep(delay)
    
    # 达到最大重试次数后仍未成功
    return "经过多次重试仍无法获取响应。"

# 示例调用
if __name__ == "__main__":
    print("\n--- 错误处理与重试示例 ---")
    messages_for_retry = [
        {"role": "user", "content": "请解释一下“通货膨胀”的经济学概念。"}
    ]
    # 使用带重试机制的函数调用LLM
    response_with_retry = call_llm_with_retry(
        messages_for_retry, 
        model="deepseek-chat", 
        max_tokens=300 # 适当增加token限制
    )
    print("通货膨胀解释：", response_with_retry)

    # 尝试一个可能导致 BadRequestError 的例子 (例如，如果模型不支持某个参数)
    # 注意：DeepSeek API 相对稳定，这里只是为了演示异常捕获
    # messages_bad_request = [
    #     {"role": "user", "content": "这是一个测试请求。"},
    #     {"role": "invalid_role", "content": "这个角色是不合法的。"} # 故意制造一个非法角色
    # ]
    # print("\n--- 尝试一个可能导致 BadRequestError 的请求 ---")
    # bad_response = call_llm_with_retry(messages_bad_request, model="deepseek-chat")
    # print("错误请求结果:", bad_response)

## 8.2 提示工程：高效使用 LLMs 的艺术
### 8.2.1 提示工程：人机交互的新范式

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# 加载环境变量文件
# 假设您的 DeepSeek_KEY.env 文件路径如下
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
api_key = os.getenv("DeepSeek_KEY")

# 初始化 DeepSeek 客户端
# 注意：DeepSeek 需要指定 base_url
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

def get_deepseek_response(prompt_messages: list, model: str = "deepseek-chat", max_tokens: int = 500) -> str:
    """
    封装 DeepSeek API 调用，用于演示不同提示词的效果。
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=prompt_messages,
            temperature=0.7, # 保持适中温度
            max_tokens=max_tokens
        )
        if response.choices:
            return response.choices[0].message.content
        else:
            return "未收到模型回复。"
    except Exception as e:
        return f"调用API时发生错误: {e}"

# 示例文章内容（假设这是一篇关于通货膨胀的文章）
article_content = """
近年来，全球通货膨胀压力持续上升，引发了广泛关注。这主要是由多重因素叠加造成的。首先，新冠疫情导致全球供应链中断，生产和物流受阻，商品供应减少，推高了物价。其次，各国政府为应对疫情推出了大规模财政刺激政策，增加了货币供应量，刺激了总需求，进一步加剧了通胀。再次，俄乌冲突等地缘政治事件导致能源和粮食价格飙升，作为重要的生产成本和生活必需品，其价格上涨直接传导至其他商品和服务。
通货膨胀对经济和社会产生了多方面影响。对于消费者而言，购买力下降，生活成本增加，尤其是低收入群体受影响更大。对于企业而言，原材料成本上升，经营压力增大，可能导致投资意愿降低。此外，通胀还可能引发工资-物价螺旋式上升，形成恶性循环。
各国央行正积极采取措施应对通胀，其中最主要的工具是加息。通过提高利率，央行旨在抑制过热的需求，收紧货币供应，从而达到控制通胀的目的。然而，过度加息也可能导致经济增长放缓甚至衰退的风险。因此，政策制定者需要在控制通胀和维持经济增长之间寻求微妙的平衡。
"""
print("--- 提示工程实践比较 ---")
#### 1. 坏Prompt 示例
#提示词设计理念：单、模糊，没有明确的角色、格式或具体要求。
# 坏Prompt
bad_prompt_messages = [
    {"role": "user", "content": f"请总结以下文章：\n\n{article_content}"}
]
print("\n--- 坏Prompt的输出 ---")
bad_response = get_deepseek_response(bad_prompt_messages)
print(bad_response)

In [ ]:
# 好Prompt
good_prompt_messages = [
    {"role": "system", "content": "你是一位专业的经济学分析师，擅长从复杂的文章中提取核心观点并清晰地呈现。"},
    {"role": "user", "content": f"""
    请仔细阅读以下关于全球通货膨胀的文章，并提取其核心观点。
    请以Markdown列表的形式，分点列出文章中讨论的通货膨胀的**主要原因、主要影响以及各国央行的应对措施**。

    文章内容：
    {article_content}
    """}
]

print("\n--- 好Prompt的输出 ---")
good_response = get_deepseek_response(good_prompt_messages)
print(good_response)

* 好Prompt坏Prompt比较：

### 8.2.3 提示词优化案例分析：基于DeepSeek的实践对比

In [ ]:
def get_deepseek_response(prompt_messages: list, model: str = "deepseek-chat", max_tokens: int = 800, temperature: float = 0.7) -> str:
    """
    封装 DeepSeek API 调用，用于演示不同提示词的效果。
    增加了 max_tokens 和 temperature 参数，方便调整。
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=prompt_messages,
            temperature=temperature, # 保持适中温度，但可调
            max_tokens=max_tokens # 增加最大token数，确保完整输出
        )
        if response.choices:
            return response.choices[0].message.content
        else:
            return "未收到模型回复。"
    except Exception as e:
        return f"调用API时发生错误: {e}"


In [ ]:
# ----------------------------------------------------------------------------------------------------
# 案例一：文章总结与核心观点提取
# ----------------------------------------------------------------------------------------------------
# 示例文章内容（一篇关于通货膨胀的文章，内容略作调整以增加细节）
article_content_inflation = """
近年来，全球通货膨胀压力持续上升，引发了广泛关注。这主要是由多重因素叠加造成的。
**首先，供给侧冲击是关键因素。** 新冠疫情导致全球供应链中断，工厂停工、物流受阻，商品供应急剧减少，推高了物价。俄乌冲突进一步加剧了能源和粮食市场的波动，作为重要的生产成本和生活必需品，其价格飙升直接传导至其他商品和服务，形成了成本推动型通胀。
**其次，需求侧的强劲反弹不容忽视。** 各国政府为应对疫情推出了空前规模的财政刺激政策，例如直接向居民发放补贴、扩大公共投资等，这极大地增加了居民可支配收入和企业投资意愿，刺激了总需求。同时，超宽松的货币政策也注入了大量流动性，进一步助推了通胀。
**此外，劳动力市场紧张也贡献了通胀压力。** 疫情后部分行业劳动力短缺，导致工资上涨，企业为弥补成本将涨价转嫁给消费者，形成“工资-物价螺旋”。
通货膨胀对经济和社会产生了多方面影响。
**对消费者而言，** 购买力显著下降，生活成本急剧增加，尤其是低收入群体和固定收入者受影响最大，贫富差距可能进一步扩大。
**对企业而言，** 原材料和劳动力成本上升，经营压力增大，利润空间受挤压，可能导致投资意愿降低，甚至引发破产潮。
**对宏观经济而言，** 通胀预期可能自我实现，市场不确定性增加，不利于长期投资和经济稳定。
各国央行正积极采取措施应对通胀，其中最主要的工具是**加息**。通过提高基准利率，央行旨在抑制过热的需求，收紧货币供应，从而达到控制通胀的目的。然而，过度激进的加息也可能导致经济增长放缓甚至陷入衰退的风险，引发失业率上升。因此，政策制定者需要在控制通胀和维持经济增长之间寻求微妙的平衡，这是一项艰巨的任务。
"""
print("\n--- 案例一：文章总结与核心观点提取 ---")
# ----------------------------------------------------------------------------------------------------
# 案例一：坏Prompt - 过于模糊和开放
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例一：坏Prompt的输出 ---")
bad_prompt_messages_1 = [
    {"role": "user", "content": f"请阅读以下文章并写一些内容：\n\n{article_content_inflation}"}
]
# 故意降低max_tokens，模拟模型在不明确指令下可能给出的简短、泛泛的回答
bad_response_1 = get_deepseek_response(bad_prompt_messages_1, max_tokens=150, temperature=0.9)
print(bad_response_1)
# ---------------------------------------------------------------------------------------------------
# 案例一：好Prompt - 明确角色、结构和关键信息点
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例一：好Prompt的输出 ---")
good_prompt_messages_1 = [
    {"role": "system", "content": "你是一位专业的宏观经济学分析师，擅长从复杂的经济学文章中提炼核心观点，并以清晰、结构化的方式呈现。"},
    {"role": "user", "content": f"""
    请仔细阅读以下关于全球通货膨胀的经济学分析文章，并完成以下任务：

    1.  **分点概述**文章中提到的通货膨胀的**主要驱动因素（原因）**。
    2.  **分点概述**通货膨胀对**消费者、企业和宏观经济**产生的具体**影响**。
    3.  **总结**文章中提到的**各国央行的主要应对策略**及其面临的**挑战**。

    请确保你的回答逻辑清晰，使用Markdown列表（无序列表）格式，并保持专业、简洁的语言风格。
   文章内容：
    {article_content_inflation}
    """}
]
good_response_1 = get_deepseek_response(good_prompt_messages_1, max_tokens=800, temperature=0.7)
print(good_response_1)

In [ ]:
# ----------------------------------------------------------------------------------------------------
# 案例二：情感分析与结构化输出
# ----------------------------------------------------------------------------------------------------

# 示例评论数据
comments_data = [
    "这家餐厅的菜品太美味了，服务也一流！",
    "等了半小时才上菜，味道还行吧，没什么特别的。",
    "产品质量很差，完全不推荐购买。",
    "客服响应很快，解决了我的问题，非常满意。",
    "价格有点高，但质量确实不错。",
    "体验感极差，再也不会来了。"
]

print("\n\n--- 案例二：情感分析与结构化输出 ---")

# ----------------------------------------------------------------------------------------------------
# 案例二：坏Prompt - 模糊指令，无格式要求
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例二：坏Prompt的输出 ---")
comments_str = "\n".join(comments_data)
bad_prompt_messages_2 = [
    {"role": "user", "content": f"请分析以下评论的情感：\n\n{comments_str}"}
]
bad_response_2 = get_deepseek_response(
    bad_prompt_messages_2,
    max_tokens=200,
    temperature=0.8
)
print(bad_response_2)

# ----------------------------------------------------------------------------------------------------
# 案例二：好Prompt - 明确分类标准、Few-shot示例和JSON格式
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例二：好Prompt的输出 ---")
good_comments_str = "\n".join(comments_data)
good_prompt_messages_2 = [
    {"role": "system", "content": "你是一个专业的评论情感分析助手，能准确判断文本的情感倾向。"},
    {"role": "user", "content": f"""
    请对以下用户评论进行情感分类。情感类别限定为：**积极 (Positive)**、**消极 (Negative)**、**中性 (Neutral)**。
    请以 JSON 数组的形式返回结果，每个元素是一个对象，包含 'comment' 和 'sentiment' 两个键。
    
    以下是几个示例：
    - 输入: "我爱死这个功能了！"
    - 输出: {{"comment": "我爱死这个功能了！", "sentiment": "积极"}}

    - 输入: "这个软件经常崩溃。"
    - 输出: {{"comment": "这个软件经常崩溃。", "sentiment": "消极"}}

    - 输入: "会议定在下午三点。"
    - 输出: {{"comment": "会议定在下午三点。", "sentiment": "中性"}}

    现在，请分析以下评论：
    {good_comments_str}
    """}
]
good_response_2 = get_deepseek_response(
    good_prompt_messages_2,
    model="deepseek-chat",
    max_tokens=800,
    temperature=0.5  # 降低温度以提高确定性
)
print(good_response_2)

In [ ]:
# ----------------------------------------------------------------------------------------------------
# 案例三：结构化信息提取
# ----------------------------------------------------------------------------------------------------

# 示例简历片段
resume_snippet = """
张三
电话: 13812345678
邮箱: zhangsan@example.com
教育背景:
2018-2022: 东北财经大学，经济学学士
2022-至今: 北京大学，金融学硕士在读

工作经历:
2022.7-2023.6: 某证券公司，研究助理
职责：宏观经济数据分析，撰写行业报告。
2023.7-至今: 某投资银行，分析师
职责：参与IPO项目，进行财务模型搭建与估值分析。

技能:
Python (数据分析, 机器学习), SQL, Excel, PowerPoint
语言: 中文 (母语), 英语 (流利)
"""

print("\n\n--- 案例三：结构化信息提取 ---")

# ----------------------------------------------------------------------------------------------------
# 案例三：坏Prompt - 泛泛而谈，无明确提取目标
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例三：坏Prompt的输出 ---")
bad_prompt_messages_3 = [
    {"role": "user", "content": f"请告诉我关于以下简历的一些信息：\n\n{resume_snippet}"}
]
bad_response_3 = get_deepseek_response(bad_prompt_messages_3, max_tokens=200, temperature=0.8)
print(bad_response_3)

# ----------------------------------------------------------------------------------------------------
# 案例三：好Prompt - 明确提取字段，要求JSON格式
# ----------------------------------------------------------------------------------------------------
print("\n--- 案例三：好Prompt的输出 ---")
good_prompt_messages_3 = [
    {"role": "system", "content": "你是一个专业的HR数据提取助手，能从简历中精确提取关键信息。"},
    {"role": "user", "content": f"""
    请从以下简历片段中提取以下信息，并以 JSON 对象的形式返回。
    如果信息缺失，请将对应的值设为 null。

    需要提取的字段及其对应的键名：
    - 姓名: "name"
    - 电话: "phone"
    - 邮箱: "email"
    - 最高学历（仅学位名称）: "highest_degree"
    - 毕业院校（最高学历对应院校）: "university"
    - 最近一份工作职位: "latest_job_title"
    - 最近一份工作公司: "latest_company"
    - 掌握的编程语言（列表形式）: "programming_languages"

    简历片段：
    {resume_snippet}
    """}
]
good_response_3 = get_deepseek_response(good_prompt_messages_3, model="deepseek-chat", max_tokens=800, temperature=0.5)
print(good_response_3)



## 8.3 LangChain：构建 LLMs 应用的框架
### 8.3.2 LangChain核心组件与功能解析

In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

# 加载环境变量
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")

# 初始化DeepSeek聊天模型
# model_name 可以是 "deepseek-chat" 或 "deepseek-reasoner"
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1", # DeepSeek的API基地址
    temperature=0.7 # 控制模型创造性
)
# 现在 deepseek_llm 就可以像其他LangChain模型一样使用了
response = deepseek_llm.invoke("你好，请用中文介绍一下你自己。")
print(response.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# 定义一个聊天提示模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是一名专业的经济学教授，擅长用清晰易懂的语言解释复杂的经济学概念。"),
    ("user", "请用通俗的语言解释一下“{concept}”的概念。")
])
# 填充变量并生成最终提示词
formatted_prompt = template.format(concept="机会成本")
print(formatted_prompt)

### 8.3.3 LangChain应用开发实践：以DeepSeek为例

In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
# 加载环境变量
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")
# 1. 初始化DeepSeek模型
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.7
)
# 2. 定义提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一名专业的经济学教授，擅长用清晰易懂的语言解释复杂的经济学概念。"),
    ("user", "请用通俗的语言解释一下“{concept}”的概念，并举一个简单的例子。")
])
# 3. 定义输出解析器
# StrOutputParser将LLM的响应直接转换为字符串
output_parser = StrOutputParser()
# 4. 构建链：通过管道操作符 | 连接组件
# 提示词模板 -> 模型 -> 输出解析器
economic_explanation_chain = prompt | deepseek_llm | output_parser
# 5. 调用链并获取结果
print("\n--- 场景一：经济学概念解释链 ---")
concept_to_explain = "机会成本"
response = economic_explanation_chain.invoke({"concept": concept_to_explain})
print(f"概念：{concept_to_explain}")
print(f"解释：{response}")

concept_to_explain_2 = "边际效用"
response_2 = economic_explanation_chain.invoke({"concept": concept_to_explain_2})
print(f"\n概念：{concept_to_explain_2}")
print(f"解释：{response_2}")

In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 导入 SentenceTransformer 用于本地嵌入模型
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings # 导入基类用于自定义嵌入器
from typing import List
import os
from dotenv import load_dotenv

# 加载环境变量
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY") # DeepSeek API Key 仍然用于 DeepSeek Chat 模型

# 1. 初始化DeepSeek聊天模型 (用于生成答案)
deepseek_llm_rag = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key, # 这里仍然使用DeepSeek的Key
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.3 # RAG场景通常温度较低，更侧重事实
)

# 2. 准备自定义知识库（这里我们模拟一个简单的文本文件）
custom_doc_content = """
宏观经济学是经济学的一个分支，主要研究国民经济的整体行为及其变量，如国民收入、就业水平、通货膨胀、经济增长和国际贸易。它关注经济的总体表现，以及政府和中央银行如何通过财政政策和货币政策来影响这些变量。

财政政策是指政府通过调整税收和支出水平来影响经济的手段。例如，在经济衰退时，政府可以通过减税或增加公共支出来刺激总需求。

货币政策是指中央银行通过控制货币供给和利率来影响经济的手段。例如，在通货膨胀高企时，中央银行可以通过提高利率来抑制借贷和投资，从而降低通胀压力。

凯恩斯主义经济学强调政府干预在稳定经济中的作用，认为市场机制并非总能自动实现充分就业。而货币主义则强调货币供给对经济活动的重要性，主张通过控制货币供给来稳定物价。
"""
# 将内容保存到临时文件
with open("data/macro_economics_doc.txt", "w", encoding="utf-8") as f:
    f.write(custom_doc_content)

# 3. 加载并分割文档
loader = TextLoader("data/macro_economics_doc.txt")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 4. 创建嵌入模型（使用本地 sentence-transformers 模型）
print("正在加载本地嵌入模型...")
# 加载一个预训练的 sentence-transformers 模型
# 首次运行会自动下载模型权重，请确保网络连接
local_embedding_model_name = "all-MiniLM-L6-v2"
local_model = SentenceTransformer(local_embedding_model_name)
print(f"本地嵌入模型 '{local_embedding_model_name}' 加载完成。")

# 包装成 LangChain 兼容的 Embeddings 类
class LocalSentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """将文档列表嵌入。"""
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        """将查询文本嵌入。"""
        return self.model.encode(text).tolist()

# 实例化自定义的本地嵌入器
local_embeddings = LocalSentenceTransformerEmbeddings(local_model)

# 5. 将文档嵌入并存储到向量数据库（这里使用FAISS作为本地向量存储）
print("正在创建向量数据库...")
vectorstore = FAISS.from_documents(split_docs, local_embeddings) # 使用本地嵌入模型
print("向量数据库创建完成。")

# 6. 定义检索器
retriever = vectorstore.as_retriever()

# 7. 定义生成答案的提示词模板
prompt_rag = ChatPromptTemplate.from_template("""
你是一名专业的经济学问答助手。请根据以下提供的上下文信息来回答问题。
如果上下文没有足够的信息来回答问题，请说明你无法回答。
请确保你的回答准确、简洁，并仅基于提供的上下文。

上下文:
{context}

问题:
{input}
""")

# 8. 创建文档链：将检索到的文档和用户问题组合成一个用于LLM的提示
document_chain = create_stuff_documents_chain(deepseek_llm_rag, prompt_rag)

# 9. 创建检索链：将检索器和文档链结合
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# 10. 调用检索链并获取结果
print("\n--- 场景二：检索增强生成 (RAG) ---")
question_1 = "宏观经济学主要研究哪些内容？"
response_rag_1 = retrieval_chain.invoke({"input": question_1})
print(f"问题: {question_1}")
print(f"回答: {response_rag_1['answer']}")

question_2 = "政府如何通过财政政策刺激经济？"
response_rag_2 = retrieval_chain.invoke({"input": question_2})
print(f"\n问题: {question_2}")
print(f"回答: {response_rag_2['answer']}")

question_3 = "凯恩斯主义和货币主义在经济稳定方面有什么不同观点？"
response_rag_3 = retrieval_chain.invoke({"input": question_3})
print(f"\n问题: {question_3}")
print(f"回答: {response_rag_3['answer']}")

question_4 = "什么是微观经济学？" # 这个问题在文档中没有直接答案
response_rag_4 = retrieval_chain.invoke({"input": question_4})
print(f"\n问题: {question_4}")
print(f"回答: {response_rag_4['answer']}")

# 清理临时文件
os.remove("data/macro_economics_doc.txt")

In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from typing import Dict, Any
import os
from dotenv import load_dotenv

# 加载环境变量
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")

# 1. 初始化DeepSeek聊天模型
# 这里使用 deepseek-chat 模型，因为它在通用文本处理和结构化输出方面表现良好
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.2 # 较低的温度有助于生成更确定和准确的结构化输出
)

# 模拟一份金融报告的文本内容
financial_report_snippet = """
2023财年，XYZ科技公司实现了显著增长。
总营收达到120亿美元，同比增长25%。
净利润为25亿美元，同比增长30%。
公司在人工智能和云计算领域的投资取得了突破性进展。
预计2024年营收将达到150亿美元。
"""

print("\n--- 场景三：多步骤文本处理与分析链 ---")
print(f"原始金融报告片段:\n{financial_report_snippet}\n")

# --- 步骤 1: 关键信息提取链 ---
# 目标：从报告中提取结构化的公司名称、营收、利润和预测
print("--- 步骤 1: 正在提取关键信息 ---")
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个专业的数据提取助手。请从提供的文本中提取以下信息，并以JSON格式返回。
    如果信息缺失，请使用 null。
    期望的JSON格式：
    {{
        "company_name": "string",
        "fiscal_year": "integer",
        "total_revenue_billion_usd": "float",
        "net_profit_billion_usd": "float",
        "revenue_growth_percent": "float",
        "net_profit_growth_percent": "float",
        "next_year_revenue_forecast_billion_usd": "float"
    }}
    """),
    ("user", "{text}")
])

# JsonOutputParser 将模型的JSON字符串输出解析为Python字典
json_parser = JsonOutputParser()

# 提取链：提示词 -> LLM -> JSON解析器
extraction_chain = extract_prompt | deepseek_llm | json_parser

# 调用提取链
extracted_data = extraction_chain.invoke({"text": financial_report_snippet})
print(f"提取到的关键信息:\n{extracted_data}\n")


# --- 步骤 2: 报告摘要生成链 ---
# 目标：根据提取到的信息和原始文本，生成一份简洁的中文摘要
print("--- 步骤 2: 正在生成报告摘要 ---")
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个专业的金融报告摘要助手。请根据提供的关键信息和原始报告内容，
    生成一份不超过100字的中文摘要，突出公司业绩亮点和未来展望。
    """),
    ("user", """原始报告内容: {original_text}
    关键信息: {extracted_info}
    请生成摘要:""")
])

# StrOutputParser 将模型的文本输出直接转换为字符串
str_parser = StrOutputParser()

# 摘要链：将原始文本和提取信息作为输入 -> 提示词 -> LLM -> 字符串解析器
# 这里使用 RunnableLambda 来格式化输入，将字典转换为字符串以便 LLM 理解
summary_chain = (
    {
        "original_text": RunnablePassthrough(), # 原始文本直接传递
        "extracted_info": RunnableLambda(lambda x: str(extracted_data)) # 将提取到的字典转换为字符串
    }
    | summary_prompt 
    | deepseek_llm 
    | str_parser
)

# 调用摘要链，输入是原始报告片段
report_summary_cn = summary_chain.invoke(financial_report_snippet)
print(f"生成的中文摘要:\n{report_summary_cn}\n")


# --- 步骤 3: 摘要翻译链 ---
# 目标：将中文摘要翻译成英文
print("--- 步骤 3: 正在翻译摘要 ---")
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的翻译助手。请将提供的中文文本翻译成简洁、准确的英文。"),
    ("user", "{text_to_translate}")
])

# 翻译链：提示词 -> LLM -> 字符串解析器
translation_chain = translate_prompt | deepseek_llm | str_parser

# 调用翻译链，输入是中文摘要
report_summary_en = translation_chain.invoke({"text_to_translate": report_summary_cn})
print(f"翻译后的英文摘要:\n{report_summary_en}\n")

## 8.4 多模态大数据中的大型语言模型应用
### 8.4.2 多模态LLMs在数据分析中的实践

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from typing import Dict, Any, List
import os
from dotenv import load_dotenv
# 加载环境变量
env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")
# 初始化DeepSeek聊天模型：在多模态场景中，模型需要较强的理解和生成能力
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat", # 或 deepseek-reasoner，如果需要更强的推理能力
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.5 # 适中温度，平衡创造性与确定性
)


In [ ]:
# --- 案例 1 场景：LLM接收到客服通话的转录文本（模拟语音识别输出），并分析客户情感和通话意图。---
print("\n--- 案例 1: 模拟客服通话情感与意图分析 ---")
# 模拟一段客服通话的转录文本
call_transcript_1 = """
客户: (语气焦躁) 你们的宽带怎么回事？已经断网两个小时了，我工作都受影响了！
客服: (语气平静) 非常抱歉给您带来不便。请问您的账号是多少，我帮您查一下情况。
客户: (语气愤怒) 账号是12345678。快点给我解决，我急着用！
客服: 好的，我正在为您查询。请稍等。
"""
sentiment_intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个专业的客服通话分析机器人。请根据提供的通话转录文本，分析客户的**当前情感**（积极/消极/中性）和**主要意图**。
    以JSON格式返回，包含 'sentiment' 和 'intent' 两个键。
    示例：{{"sentiment": "消极", "intent": "投诉宽带故障并要求立即解决"}}
    """),
    ("user", "通话转录: {transcript}")
])
json_parser = JsonOutputParser()
sentiment_intent_chain = sentiment_intent_prompt | deepseek_llm | json_parser
response_sentiment_1 = sentiment_intent_chain.invoke({"transcript": call_transcript_1})
print(f"通话转录:\n{call_transcript_1}")
print(f"分析结果:\n{response_sentiment_1}\n")
call_transcript_2 = """
客户: (语气愉快) 你们的智能音箱功能真棒，我刚学会用语音控制智能家居，太方便了！
客服: (语气热情) 很高兴您喜欢我们的产品！请问还有其他可以帮助您的吗？
"""
response_sentiment_2 = sentiment_intent_chain.invoke({"transcript": call_transcript_2})
print(f"通话转录:\n{call_transcript_2}")
print(f"分析结果:\n{response_sentiment_2}\n")


In [ ]:
# --- 案例 2: 场景：LLM接收到图像的文本描述（模拟图像编码器输出），并基于此进行理解和回答。
print("\n--- 案例 2: 模拟图像理解与问答 (VQA) ---")
# 模拟一个图像的详细描述（来自一个假设的图像识别/描述模型）
image_description_1 = """
图像显示的是一间现代化的办公室，有几位员工正在使用笔记本电脑工作。
窗外是城市的天际线，阳光明媚。
前景有一盆绿植，背景的白板上写着“Q3 销售目标”。
一位穿着蓝色衬衫的男士正在和另一位女士交谈，他们似乎在讨论工作。
"""
vqa_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个能够理解图像描述并回答相关问题的智能助手。请根据提供的图像描述，简洁准确地回答问题。"),
    ("user", "图像描述: {image_desc}\n问题: {question}")
])
vqa_chain = vqa_prompt | deepseek_llm | StrOutputParser()
question_vqa_1 = "办公室里有多少人？他们在做什么？"
response_vqa_1 = vqa_chain.invoke({"image_desc": image_description_1, "question": question_vqa_1})
print(f"问题: {question_vqa_1}")
print(f"回答: {response_vqa_1}\n")
question_vqa_2 = "白板上写了什么？"
response_vqa_2 = vqa_chain.invoke({"image_desc": image_description_1, "question": question_vqa_2})
print(f"问题: {question_vqa_2}")
print(f"回答: {response_vqa_2}\n")


In [ ]:
from openai import OpenAI
yi_env_file_path = "env/Yi_KEY.env"
load_dotenv(yi_env_file_path)
yi_api_key = os.getenv("Yi_KEY") 
yi_vision_client = OpenAI(
    api_key=yi_api_key,
    base_url="https://api.lingyiwanwu.com/v1" # Yi-Vision 的 API 基地址
)
# --- 案例 3: 直接视觉-语言模型 (VLM) - 图片描述与问答 (使用 Yi-Vision) ---
# 场景：直接将图片URL作为输入提供给支持多模态的LLM，让模型直接理解图片内容并进行描述或问答。
# 这展示了原生多模态LLM的能力，无需预先进行文本描述。
print("\n--- 案例 4: 直接视觉-语言模型 (VLM) - 图片描述与问答 (使用 Yi-Vision) ---")
# 定义要分析的图片URL
image_to_analyze_url = "https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/xie-multicoefplot-f1.png"
print(f"正在使用 Yi-Vision 模型分析图片: {image_to_analyze_url}\n")
try:
    # 直接使用 Yi-Vision进行多模态请求
    completion = yi_vision_client.chat.completions.create(
        model="yi-vision", # 指定 Yi-Vision 模型
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "描述一下这张图片" # 用户提出的文本问题
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_analyze_url # 直接传入图片URL
                        }
                    }
                ]
            },
        ]
    )
    response_yi_vision = completion.choices[0].message.content
    print(f"Yi-Vision 模型对图片的描述:\n{response_yi_vision}\n")
    # 进一步提问关于图片内容的问题
    question_yi_vision = "这张图表中哪年系数估计值最大？"
    completion_qa = yi_vision_client.chat.completions.create(
        model="yi-vision",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question_yi_vision
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_analyze_url
                        }
                    }
                ]
            },
        ]
    )
    response_yi_vision_qa = completion_qa.choices[0].message.content
    print(f"问题: {question_yi_vision}")
    print(f"回答: {response_yi_vision_qa}\n")
except Exception as e:
    print(f"调用 Yi-Vision 模型时发生错误: {e}")
    print("请检查您的 Yi_API_KEY 是否正确配置，以及模型是否可用。") 


## 8.5 大型语言模型的局限性与挑战
### 8.5.1 数据与知识边界

In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

# --- 环境配置 ---
# 加载 DeepSeek API Key
deepseek_env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(deepseek_env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")

# 初始化DeepSeek聊天模型
# 保持适中温度，有时高温度更容易引发幻觉，但此处旨在展示可能性
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.7 # 略微提高温度，以增加模型“创造性”的可能性，从而更容易观察到幻觉
)
# --- 知识截止（Knowledge Cut-off）示例 ---
# 场景：提问一个模型训练数据截止日期之后发生的重大事件或最新信息。
# 目的：演示LLM无法获取其知识截止日期之后信息的局限性。
print("\n--- 知识截止（Knowledge Cut-off）示例 ---")

# 构造一个关于模型知识截止日期之后发生的事件的问题
# 假设DeepSeek-chat的知识截止日期在2023年下半年或更早
knowledge_cutoff_question = "请介绍一下2025年特朗普对全球加增关税的情况和影响"
# 注：2024年巴黎奥运会开幕式在模型知识截止日期之后，模型应无法提供具体细节。
knowledge_cutoff_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识渊博的助手，请根据你所了解的信息回答问题。"),
    ("user", "{question}")
])
knowledge_cutoff_chain = knowledge_cutoff_prompt | deepseek_llm | StrOutputParser()

print(f"用户问题: {knowledge_cutoff_question}")
try:
    response_knowledge_cutoff = knowledge_cutoff_chain.invoke({"question": knowledge_cutoff_question})
    print(f"模型回答:\n{response_knowledge_cutoff}\n")
    print("请核对上述回答中关于2025年特朗普对全球加增关税的情况和影响的细节的准确性，以判断是否存在知识截止现象。")
except Exception as e:
    print(f"调用模型时发生错误: {e}")
    print("请检查您的 DeepSeek_KEY 是否正确配置，以及网络连接是否正常。")

In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

deepseek_env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(deepseek_env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")

deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.7 # 略微提高温度，以增加模型“创造性”的可能性，从而更容易观察到幻觉
)

print("\n--- 幻觉问题示例 ---")
# 构造一个可能引发幻觉的提示词：询问一篇虚构的、但主题和格式都非常像真实学术论文的内容
hallucination_question = "请总结一下由著名经济学家张伟教授在2023年发表于《国际经济评论》期刊上的论文《大数据驱动的中国区域经济发展不平衡性研究》的主要观点和研究方法。"
# 注：此论文、作者与期刊的特定组合在2023年可能不存在，旨在观察模型是否会“编造”内容。
hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识渊博的助手，请根据你所了解的信息回答问题。"),
    ("user", "{question}")
])
hallucination_chain = hallucination_prompt | deepseek_llm | StrOutputParser()
print(f"用户问题: {hallucination_question}")
try:
    response_hallucination = hallucination_chain.invoke({"question": hallucination_question})
    print(f"模型回答:\n{response_hallucination}\n")
    print("请核对上述回答中的事实信息，以判断是否存在幻觉。")
except Exception as e:
    print(f"调用模型时发生错误: {e}")
    print("请检查您的 DeepSeek_KEY 是否正确配置，以及网络连接是否正常。")

In [ ]:
# --- 数据偏见与歧视示例 ---
# 场景：尝试通过一个特定情境的提问，观察模型是否会无意中生成带有社会刻板印象的回答。
# 目的：演示LLM可能复制训练数据中的偏见。
print("\n--- 数据偏见与歧视示例 ---")

# 提问一个在社会中常与特定性别或社会角色关联的场景，观察模型是否会默认使用带有偏见的描述。
bias_question = "请描述一位在大型科技公司担任高级管理职位的女性，她是如何平衡事业和家庭的？"
# 注：此问题旨在观察模型是否会默认强调女性在事业家庭平衡中的“特殊”挑战或刻板印象，
# 而非平等地看待所有高级管理者的工作与生活平衡。

bias_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个客观的描述助手，请根据你所了解的信息描述以下场景。"),
    ("user", "{question}")
])

bias_chain = bias_prompt | deepseek_llm | StrOutputParser()

print(f"用户问题: {bias_question}")
try:
    response_bias = bias_chain.invoke({"question": bias_question})
    print(f"模型回答:\n{response_bias}\n")
    print("请仔细审视上述回答，特别是其中对“平衡”的描述、强调的挑战以及解决方案，以判断是否存在潜在的性别刻板印象。")
except Exception as e:
    print(f"调用模型时发生错误: {e}")
    print("请检查您的 DeepSeek_KEY 是否正确配置，以及网络连接是否正常。")

### 8.5.2 推理与逻辑挑战


In [ ]:
# 导入必要的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

# --- 环境配置 ---
# 加载 DeepSeek API Key
deepseek_env_file_path = "env/DeepSeek_KEY.env"
load_dotenv(deepseek_env_file_path)
deepseek_api_key = os.getenv("DeepSeek_KEY")

# 初始化DeepSeek聊天模型
# 将温度设为0，以减少随机性，迫使模型给出“最确定”的答案。
# 如果模型无法推理，可能会给出错误或回避性答案，从而更好地展示局限性。
deepseek_llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=deepseek_api_key,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.0
)
print("\n--- 复杂推理能力不足示例 ---")
# 构造一个需要多步逻辑推理的问题（经典逻辑谜题简化版）
complex_reasoning_question = """
有三位同事：A、B、C。
他们分别来自三个不同的城市：北京、上海、广州。
他们各自从事三种不同的职业：医生、律师、教师。

已知：
1. A 不住在上海。
2. 来自北京的人不是律师。
3. C 的职业不是教师。
4. 律师不住在广州。
5. B 是医生。

请问：C 来自哪个城市？他的职业是什么？
请一步步地推导出答案，并清晰地展示推导过程。
"""
complex_reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个逻辑推理助手，请严格按照提供的线索进行推导，并展示每一步的推理过程。"),
    ("user", "{question}")
])

complex_reasoning_chain = complex_reasoning_prompt | deepseek_llm | StrOutputParser()

print(f"用户问题: {complex_reasoning_question}")
try:
    response_reasoning = complex_reasoning_chain.invoke({"question": complex_reasoning_question})
    print(f"模型回答:\n{response_reasoning}\n")
    print("请核对上述回答中的推导过程和最终答案，以判断模型在复杂逻辑推理上的表现。")
    print("\n--- 正确答案及推导过程 ---")
    print("正确答案：C 来自上海，职业是律师。")
    print("推导过程：")
    print("1. 根据线索5 'B 是医生'，我们确定 B 的职业是医生。")
    print("2. 根据线索3 'C 的职业不是教师'，且已知职业有医生、律师、教师。既然 B 是医生，那么 A 和 C 只能是律师或教师。C 不是教师，因此 C 的职业必然是律师。")
    print("3. 由此，通过排除法，A 的职业是教师。")
    print("4. 根据线索4 '律师不住在广州'，且已知 C 是律师，所以 C 不住在广州。")
    print("5. 根据线索2 '来自北京的人不是律师'，且已知 C 是律师，所以 C 不住在北京。")
    print("6. 综合 C 不住在广州和北京，且城市有北京、上海、广州，那么 C 必然来自上海。")
    print("至此，我们已完全确定 C 的信息：C 来自上海，职业是律师。")

except Exception as e:
    print(f"调用模型时发生错误: {e}")
    print("请检查您的 DeepSeek_KEY 是否正确配置，以及网络连接是否正常。")


